# 03. FVI / FDI Calculation (Reproduction Notebook)

## 1. Data Sources

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# 노트북 파일 위치를 기준으로 한 상대경로 (절대경로 미사용)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# GitHub 정리 작업 이후 영어 snake_case 파일명으로 갱신됨 (원본 한글 파일명은 감사 폴더에 보존)
PATH_2020 = RAW_DIR / "district_indicators_2020.csv"
PATH_2022 = RAW_DIR / "district_indicators_2022.csv"
PATH_REFERENCE = RAW_DIR / "fvi_presentation_source.xlsx"

df_2020 = pd.read_csv(PATH_2020, encoding="utf-8")
df_2022 = pd.read_csv(PATH_2022, encoding="utf-8")

print("2020:", df_2020.shape)
print("2022:", df_2022.shape)


2020: (25, 21)
2022: (25, 21)


## 2. Indicator Definitions

**Group 1 — 과거 홍수 피해 규모 (과거피해점수)**
- 침수가구비율 = 침수세대 수 (세대) / 인구
- 단위면적당피해규모 (천원/km²)
- 피해인구비율 = 피해인구 / 인구

**Group 2 — 홍수에 취약한 지역 특성 (지역특성점수)**
- 인구밀도 (명/㎢)
- 불투수면적 비율(퍼센트)
- 가구 밀도(퍼센트)

**Group 3 — 홍수 대응 인프라 및 회복력 (인프라점수, 값이 클수록 취약도가 낮아지므로 역정규화)**
- 하수관거 길이
- 재정자립도
- 소방인력 수

**FVI (Flood Vulnerability Index)**
`fvi = 과거피해점수 * w_g1 + 지역특성점수 * w_g2 + 인프라점수 * w_g3`

**FDI (Flood Defense Infrastructure Index, 원본 변수명: 방어용량)**
원본 노트북은 `방어용량`이라는 이름으로 방어 인프라 지표를 계산합니다. 이 연구의 최종 분석 흐름에서
FDI에 해당하는 값으로 판단하여 그대로 사용했습니다 (근거는 9번 섹션에 기록).

- 외수방어능력 (m) / max
- 내수방어능력 (m³/min) / max
- 방어시설용량 (h) / max
- `fdi = (외수_norm + 내수_norm + 시설_norm) / 3`  (원본 코드 그대로, 3요소 단순 평균)


## 3. Data Validation

In [ ]:
REQUIRED_COLUMNS = [
    "자치구",
    "침수세대 수 (세대)", "인구", "피해인구", "단위면적당피해규모 (천원/km²)",
    "인구밀도 (명/㎢)", "불투수면적 비율(퍼센트)", "가구 밀도(퍼센트)",
    "하수관거 길이", "재정자립도", "소방인력 수",
    "외수방어능력 (m)", "내수방어능력 (m³/min)", "방어시설용량 (h)",
]

validation_report = {}
for label, df in [("2020", df_2020), ("2022", df_2022)]:
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    null_counts = df[REQUIRED_COLUMNS].isnull().sum()
    validation_report[label] = {
        "n_rows": len(df),
        "n_districts_unique": df["자치구"].nunique(),
        "missing_columns": missing_cols,
        "columns_with_nulls": null_counts[null_counts > 0].to_dict(),
    }
    assert len(df) == 25, f"{label}: 자치구 수가 25개가 아닙니다 ({len(df)}개)"
    assert not missing_cols, f"{label}: 누락된 컬럼 {missing_cols}"

validation_report


{'2020': {'n_rows': 25,
  'n_districts_unique': 25,
  'missing_columns': [],
  'columns_with_nulls': {}},
 '2022': {'n_rows': 25,
  'n_districts_unique': 25,
  'missing_columns': [],
  'columns_with_nulls': {}}}

## 4. Normalization

In [ ]:
def safe_normalize(series: pd.Series) -> pd.Series:
    """합계 정규화 (분모가 0이면 전부 0 반환)."""
    total = series.sum()
    return series / total if total != 0 else series * 0

def safe_inverse_normalize(series: pd.Series) -> pd.Series:
    """정규화 후 역수 처리 (0 값은 0으로 유지)."""
    norm = safe_normalize(series)
    inv = pd.Series(np.zeros(len(series)), index=series.index)
    inv[norm != 0] = 1 / norm[norm != 0]
    return inv


def build_indicator_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df[["자치구"]].copy()

    # 침수가구비율 / 피해인구비율 (인구 기반 파생 변수)
    out["침수가구비율"] = df["침수세대 수 (세대)"] / df["인구"]
    out["피해인구비율"] = df["피해인구"] / df["인구"]

    # Group 1: 과거 피해 규모
    out["침수가구비율_정규화"] = safe_normalize(out["침수가구비율"])
    out["단위면적당피해규모_정규화"] = safe_normalize(df["단위면적당피해규모 (천원/km²)"])
    out["피해인구비율_정규화"] = safe_normalize(out["피해인구비율"])

    # Group 2: 지역 특성
    out["인구밀도_정규화"] = safe_normalize(df["인구밀도 (명/㎢)"])
    out["불투수면적_정규화"] = safe_normalize(df["불투수면적 비율(퍼센트)"])
    out["가구밀도_정규화"] = safe_normalize(df["가구 밀도(퍼센트)"])

    # Group 3: 인프라 (역정규화 - 값이 클수록 취약도 낮음)
    out["하수관길이_역정규화"] = safe_inverse_normalize(df["하수관거 길이"])
    out["재정자립도_역정규화"] = safe_inverse_normalize(df["재정자립도"])
    out["소방공무원수_역정규화"] = safe_inverse_normalize(df["소방인력 수"])

    return out

indicators_2020 = build_indicator_frame(df_2020)
indicators_2022 = build_indicator_frame(df_2022)

indicators_2022.head()


,자치구,침수가구비율,피해인구비율,침수가구비율_정규화,단위면적당피해규모_정규화,피해인구비율_정규화,인구밀도_정규화,불투수면적_정규화,가구밀도_정규화,하수관길이_역정규화,재정자립도_역정규화,소방공무원수_역정규화
0,동작구,0.010089,0.016766,0.212288,0.279031,0.214961,0.056799,0.045910,0.042278,32.378678,26.864769,31.670886
1,영등포구,0.010412,0.018775,0.219096,0.168409,0.240717,0.038584,0.045581,0.044091,22.013698,20.238606,26.153310
2,관악구,0.009602,0.014071,0.202054,0.149426,0.180414,0.040335,0.029967,0.063844,25.109631,38.515306,27.394161
3,강남구,0.002121,0.003692,0.044636,0.053606,0.047338,0.032176,0.041972,0.050837,14.831347,12.816638,20.564384
4,서초구,0.002725,0.004857,0.057338,0.140008,0.062278,0.020692,0.027665,0.036892,18.015648,13.060554,23.603774


## 5. AHP Weights

In [ ]:
GROUP_WEIGHTS = {
    "과거피해점수": 0.2321,   # 그룹1(과거 홍수 피해 규모) 그룹간 중요도
    "지역특성점수": 0.4274,   # 그룹2(홍수에 취약한 지역 특성) 그룹간 중요도
    "인프라점수": 0.3405,     # 그룹3(홍수 대응 인프라 및 회복력) 그룹간 중요도
}

GROUP1_WEIGHTS = {  # 과거 홍수 피해 규모 내부 가중치
    "침수가구비율_정규화": 0.2595,
    "단위면적당피해규모_정규화": 0.2431,
    "피해인구비율_정규화": 0.4974,
}

GROUP2_WEIGHTS = {  # 홍수에 취약한 지역 특성 내부 가중치
    "인구밀도_정규화": 0.2250,
    "불투수면적_정규화": 0.4157,
    "가구밀도_정규화": 0.3592,
}

GROUP3_WEIGHTS = {  # 홍수 대응 인프라 및 회복력 내부 가중치
    "하수관길이_역정규화": 0.3824,
    "재정자립도_역정규화": 0.3390,
    "소방공무원수_역정규화": 0.2786,
}

# 원본 노트북의 가중치 상수가 소수 4자리로 반올림되어 있어 합계가 1.0에서 최대 1e-3 수준 벗어남
assert abs(sum(GROUP_WEIGHTS.values()) - 1.0) < 1e-3
assert abs(sum(GROUP1_WEIGHTS.values()) - 1.0) < 1e-3
assert abs(sum(GROUP2_WEIGHTS.values()) - 1.0) < 1e-3
assert abs(sum(GROUP3_WEIGHTS.values()) - 1.0) < 1e-3

GROUP_WEIGHTS, GROUP1_WEIGHTS, GROUP2_WEIGHTS, GROUP3_WEIGHTS


({'과거피해점수': 0.2321, '지역특성점수': 0.4274, '인프라점수': 0.3405},
 {'침수가구비율_정규화': 0.2595, '단위면적당피해규모_정규화': 0.2431, '피해인구비율_정규화': 0.4974},
 {'인구밀도_정규화': 0.225, '불투수면적_정규화': 0.4157, '가구밀도_정규화': 0.3592},
 {'하수관길이_역정규화': 0.3824, '재정자립도_역정규화': 0.339, '소방공무원수_역정규화': 0.2786})

## 7. FVI Calculation

`fvi = 과거피해점수 * 0.2321 + 지역특성점수 * 0.4274 + 인프라점수 * 0.3405` 


In [ ]:
def compute_fvi(indicators: pd.DataFrame) -> pd.DataFrame:
    result = indicators[["자치구"]].copy()

    result["과거피해점수"] = sum(
        indicators[col] * w for col, w in GROUP1_WEIGHTS.items()
    )
    result["지역특성점수"] = sum(
        indicators[col] * w for col, w in GROUP2_WEIGHTS.items()
    )
    result["인프라점수"] = sum(
        indicators[col] * w for col, w in GROUP3_WEIGHTS.items()
    )

    result["fvi"] = (
        result["과거피해점수"] * GROUP_WEIGHTS["과거피해점수"]
        + result["지역특성점수"] * GROUP_WEIGHTS["지역특성점수"]
        + result["인프라점수"] * GROUP_WEIGHTS["인프라점수"]
    )
    return result

fvi_2020 = compute_fvi(indicators_2020)
fvi_2022 = compute_fvi(indicators_2022)

fvi_2022.sort_values("fvi", ascending=False).head()


,자치구,과거피해점수,지역특성점수,인프라점수,fvi
8,금천구,0.097421,0.039482,37.015316,12.643201
24,강북구,0.012168,0.029553,35.281262,12.028725
23,도봉구,0.006731,0.031980,34.727337,11.839889
16,중랑구,0.000257,0.044541,32.271025,11.007381
20,노원구,0.002745,0.036905,32.065317,10.934651


## 8. FDI Calculation

In [ ]:
def compute_fdi(df: pd.DataFrame) -> pd.DataFrame:
    result = df[["자치구"]].copy()

    ext_max = df["외수방어능력 (m)"].max()
    int_max = df["내수방어능력 (m³/min)"].max()
    fac_max = df["방어시설용량 (h)"].max()

    result["fdi"] = (
        df["외수방어능력 (m)"] / ext_max
        + df["내수방어능력 (m³/min)"] / int_max
        + df["방어시설용량 (h)"] / fac_max
    ) / 3
    return result

fdi_2020 = compute_fdi(df_2020)
fdi_2022 = compute_fdi(df_2022)

fdi_2022.sort_values("fdi", ascending=False).head()


,자치구,fdi
6,송파구,0.533175
18,성동구,0.519168
23,도봉구,0.460492
22,종로구,0.459630
24,강북구,0.451030


## 9. Validation Against Final Presentation


In [ ]:
reference_2020 = pd.read_excel(PATH_REFERENCE, sheet_name="2020").rename(columns={"fvi_점수": "fvi_reference"})
reference_2022 = pd.read_excel(PATH_REFERENCE, sheet_name="2022").rename(columns={"fvi_점수": "fvi_reference"})

def compare_with_reference(computed: pd.DataFrame, reference: pd.DataFrame, label: str) -> pd.DataFrame:
    merged = computed[["자치구", "fvi"]].merge(reference, on="자치구", how="outer", indicator=True)
    merged["abs_diff"] = (merged["fvi"] - merged["fvi_reference"]).abs()
    merged["year"] = label
    return merged.sort_values("abs_diff", ascending=False)

comparison_2020 = compare_with_reference(fvi_2020, reference_2020, "2020")
comparison_2022 = compare_with_reference(fvi_2022, reference_2022, "2022")
comparison_all = pd.concat([comparison_2020, comparison_2022], ignore_index=True)

comparison_all


,자치구,fvi,fvi_reference,_merge,abs_diff,year
0,강북구,12.065796,0.024785,both,12.041011,2020
1,금천구,12.055555,0.030772,both,12.024783,2020
2,도봉구,12.032667,0.025379,both,12.007288,2020
3,노원구,11.095774,0.040773,both,11.055001,2020
4,중랑구,10.997346,0.043363,both,10.953983,2020
5,서대문구,10.621625,0.049032,both,10.572593,2020
6,광진구,10.482459,0.033959,both,10.448500,2020
7,동작구,10.452198,0.037009,both,10.415189,2020
8,성동구,10.236826,0.031798,both,10.205028,2020
9,관악구,10.197303,0.034631,both,10.162672,2020


In [ ]:
summary = comparison_all.groupby("year")["abs_diff"].agg(["count", "mean", "max"])
summary


,count,mean,max
year,,,
2020,25,9.300525,12.041011
2022,25,9.337775,12.583324


## 10. Export Results


In [ ]:
final_df = (
    fvi_2020[["자치구", "fvi"]].rename(columns={"자치구": "district", "fvi": "fvi_2020"})
    .merge(fvi_2022[["자치구", "fvi"]].rename(columns={"자치구": "district", "fvi": "fvi_2022"}), on="district", how="outer")
    .merge(fdi_2020[["자치구", "fdi"]].rename(columns={"자치구": "district", "fdi": "fdi_2020"}), on="district", how="outer")
    .merge(fdi_2022[["자치구", "fdi"]].rename(columns={"자치구": "district", "fdi": "fdi_2022"}), on="district", how="outer")
)

output_path = PROCESSED_DIR / "03_fvi_fdi_results.csv"
final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved: {output_path}")
final_df


Saved: /home/dslab/shlee/flood_risk_insurance_model/data/processed/03_fvi_fdi_results.csv


,district,fvi_2020,fvi_2022,fdi_2020,fdi_2022
0,강동구,8.665200,8.802396,0.062547,0.062547
1,성북구,9.973080,10.064700,0.348732,0.348732
2,영등포구,7.676537,7.752530,0.106404,0.106404
3,송파구,6.444388,6.403339,0.533175,0.533175
4,강남구,5.538205,5.390906,0.371091,0.371091
5,서대문구,10.621625,10.463431,0.285130,0.285130
6,중구,7.882331,7.853469,0.298989,0.298989
7,중랑구,10.997346,11.007381,0.384531,0.384531
8,동대문구,9.721267,9.802705,0.285186,0.285186
9,노원구,11.095774,10.934651,0.215972,0.215972
